# Large Area Processing

The cube is too large to load into memory. These cells use Dask reductions, preserve the native Zarr chunks, and return small tables.

Use Dask Gateway for this notebook. This will not run locally unless a Dask Gateway is available.

In [6]:
import dask.array as da
import dask.distributed
import warnings
import numpy as np
import xarray as xr
from dask.array.core import PerformanceWarning
from dask_gateway import Gateway

warnings.filterwarnings("ignore", message="In a future version of xarray the default value for join.*", category=FutureWarning)
warnings.filterwarnings("ignore", message="Increasing number of chunks.*", category=PerformanceWarning)


In [1]:
## run this line if you get a file system not found error!
# import os
# os.chdir("/tmp")
# print("cwd:", os.getcwd())

In [ ]:
gateway = Gateway()
cluster_options = gateway.cluster_options()
cluster_options


In [ ]:
cluster = gateway.new_cluster(cluster_options=cluster_options)
cluster.scale(2)
cluster


In [ ]:
client = cluster.get_client()
client


In [10]:
cube_paths = [
    "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/antarctica_cube/icetemp.zarr",
    "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/antarctica_cube/sec.zarr",
    "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/antarctica_cube/antarctica-combined.zarr",
    "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/antarctica_cube/icemask_composite.zarr/",
    "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/antarctica_cube/ice_velocity.zarr",
]

ds = xr.open_mfdataset(cube_paths, engine="zarr", chunks={}, compat="no_conflicts")
ds


<xarray.Dataset> Size: 7TB
Dimensions:                                       (y: 49158, x: 57358,
                                                   depth: 91, time_period: 27,
                                                   time: 48)
Coordinates:
  * y                                             (y) float32 197kB -2.458e+0...
  * x                                             (x) float32 229kB -2.868e+0...
  * depth                                         (depth) int16 182B 0 ... 4500
  * time_period                                   (time_period) int64 216B 0 ...
  * time                                          (time) datetime64[ns] 384B ...
    spatial_ref                                   int64 8B 0
Data variables: (12/40)
    englacial_temp_profile_quality_flag           (y, x) float16 6GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    englacial_temp_profile_tice                   (depth, y, x) float16 513GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>
    surface_elevation_change_basin_id             (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    surface_elevation_change_cell_end_times       (time_period, y, x) float32 305GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>
    surface_elevation_change_cell_start_times     (time_period, y, x) float32 305GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>
    surface_elevation_change_end_time             (time_period) float32 108B dask.array<chunksize=(1,), meta=np.ndarray>
    ...                                            ...
    ice_sheet_surface_velocity_easting_stddev     (time, y, x) float32 541GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>
    ice_sheet_surface_velocity_magnitude          (time, y, x) float32 541GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>
    ice_sheet_surface_velocity_measurement_count  (time, y, x) float32 541GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>
    ice_sheet_surface_velocity_northing           (time, y, x) float32 541GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>
    ice_sheet_surface_velocity_northing_stddev    (time, y, x) float32 541GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>
    ice_sheet_surface_velocity_vertical           (time, y, x) float32 541GB dask.array<chunksize=(1, 4915, 5735), meta=np.ndarray>

In [11]:
dx = abs((ds.x.isel(x=1) - ds.x.isel(x=0)).load().item())
dy = abs((ds.y.isel(y=1) - ds.y.isel(y=0)).load().item())
km2 = dx * dy / 1e6

ds[[
    "bedrock_topography_mask",
    "bedrock_topography_bed",
    "bedrock_topography_thickness",
    "bedrock_topography_errbed",
    "ice_shelf_basal_melt_rate",
    "groundlines_mask",
    "subglacial_lakes_mask",
    "supra_glacial_lakes_mask",
]]


<xarray.Dataset> Size: 65GB
Dimensions:                       (y: 49158, x: 57358)
Coordinates:
  * y                             (y) float32 197kB -2.458e+06 ... 2.458e+06
  * x                             (x) float32 229kB -2.868e+06 ... 2.868e+06
    spatial_ref                   int64 8B 0
Data variables:
    bedrock_topography_mask       (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    bedrock_topography_bed        (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    bedrock_topography_thickness  (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    bedrock_topography_errbed     (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    ice_shelf_basal_melt_rate     (y, x) float32 11GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    groundlines_mask              (y, x) uint8 3GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    subglacial_lakes_mask         (y, x) uint8 3GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>
    supra_glacial_lakes_mask      (y, x) uint8 3GB dask.array<chunksize=(4915, 5735), meta=np.ndarray>

## Area of the main surface classes

Mask codes: 0 ocean, 1 ice-free land, 2 grounded ice, 3 floating ice, 4 Lake Vostok.

In [12]:
mask = ds["bedrock_topography_mask"]

surface_area = xr.Dataset({
    "ocean": (mask == 0).sum() * km2,
    "ice_free_land": (mask == 1).sum() * km2,
    "grounded_ice": (mask == 2).sum() * km2,
    "floating_ice": (mask == 3).sum() * km2,
    "lake_vostok": (mask == 4).sum() * km2,
}).compute()

surface_area.to_array().to_series().rename("area_km2").to_frame()


,area_km2
variable,
ocean,14602610.14
ice_free_land,70024.25
grounded_ice,11991005.25
floating_ice,1517202.00
lake_vostok,15204.00


## Example 1: marine-based grounded ice exposure

This estimates how much grounded ice sits on bed below sea level, including deep basins below -1000 m.

In [13]:
bed = ds["bedrock_topography_bed"]
thickness = ds["bedrock_topography_thickness"]
err = ds["bedrock_topography_errbed"]
marine = (mask == 2) & (bed < 0)

marine_summary = xr.Dataset({
    "grounded_ice_bed_below_0m_km2": marine.sum() * km2,
    "grounded_ice_bed_below_minus_1000m_km2": (marine & (bed < -1000)).sum() * km2,
    "mean_bed_elevation_m": bed.where(marine).mean(),
    "mean_ice_thickness_m": thickness.where(marine).mean(),
    "mean_bed_error_m": err.where(marine).mean(),
}).compute()

marine_summary.to_array().to_series().rename("value").to_frame()


,value
variable,
grounded_ice_bed_below_0m_km2,5.457388e+06
grounded_ice_bed_below_minus_1000m_km2,5.941695e+05
mean_bed_elevation_m,-5.014638e+02
mean_ice_thickness_m,2.241388e+03
mean_bed_error_m,1.246957e+02


## Example 2: floating-ice basal melt and a 5 km grounding-zone belt

The melt variable is labelled as a basal melt rate, but the metadata unit is `m`. If the source convention is m/yr, the volume columns are km³/yr.

In [14]:
from scipy.ndimage import binary_dilation

melt = ds["ice_shelf_basal_melt_rate"]
floating = (mask == 3) & melt.notnull()

gl5 = da.map_overlap(
    binary_dilation,
    ds["groundlines_mask"].data.astype(bool),
    depth=5,
    boundary=False,
    dtype=bool,
    structure=np.ones((11, 11), bool),
)
grounding_zone = xr.DataArray(gl5, coords=mask.coords, dims=mask.dims) & floating

melt_summary = xr.Dataset({
    "floating_ice_with_melt_data_km2": floating.sum() * km2,
    "floating_ice_mean_melt_m": melt.where(floating).mean(),
    "floating_ice_net_melt_km3_per_dataset_unit": melt.where(floating).sum() * km2 / 1000,
    "grounding_zone_5km_area_km2": grounding_zone.sum() * km2,
    "grounding_zone_5km_mean_melt_m": melt.where(grounding_zone).mean(),
}).compute()

melt_summary.to_array().to_series().rename("value").to_frame()


,value
variable,
floating_ice_with_melt_data_km2,1.458565e+06
floating_ice_mean_melt_m,7.076098e-01
floating_ice_net_melt_km3_per_dataset_unit,1.032095e+03
grounding_zone_5km_area_km2,2.548982e+04
grounding_zone_5km_mean_melt_m,4.801201e+00


In [ ]:
client.close()
